# Fine-tune VNUF Empathetic Assistant (QLoRA trên Qwen 4B)

Notebook này fine-tune model chat đồng cảm tiếng Việt từ 2 nguồn data:
- **Single-turn**: 3 file CSV (~5.1k cặp hỏi–đáp), gộp lại để đa dạng độ dài.
- **Multi-turn**: `hoithoaiiii.jsonl` (91 cuộc hội thoại nhiều lượt).

Quyết định đã chốt trước đó:
- **Bỏ nhãn emotion khỏi input training** (nhãn dự đoán được từ text 97%+ → redundant; classifier runtime chỉ ~59% → label leakage). Input chỉ là `messages`.
- Train bằng **QLoRA** (model 4-bit + adapter LoRA) — vừa GPU consumer, lần đầu chỉ train 1–2 epochs, early stop theo val loss.

## Cell 1 — Cài đặt thư viện

Cài các gói cần thiết. `unsloth` tối ưu tốc độ + VRAM cho QLoRA; `trl` cung cấp `SFTTrainer` cho supervised fine-tune định dạng chat; `datasets`/`pandas` để đọc data. Chạy 1 lần đầu, các lần sau có thể bỏ qua.

In [2]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"  # unsloth bản mới nhất
!pip install -q trl peft accelerate bitsandbytes datasets pandas

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 24.2 MB/s eta 0:

### Ghi nhớ khi train trên Colab T4

T4 (Colab free) **không hỗ trợ bf16** → phải train ở fp16. Đã set sẵn `dtype="fp16"` ở Cell 7. Nếu model báo về loss = NaN/inf ngay step đầu thì thử giảm LR 1e-4 hoặc đổi sang bf16 khi chuyển GPU A100/T4 Ampere+.

## Cell 2 — Cấu hình

Tập trung toàn bộ tham số thay đổi được vào 1 chỗ: đường dẫn data, tên base model, hyperparams LoRA và training. Muốn đổi model/epochs/LR chỉ sửa cell này.

`MAX_SEQ_LEN = 2048` đủ phủ hội thoại 8 lượt; nếu gặp OOM thì giảm xuống 1024 hoặc giảm `BATCH_SIZE`.

In [3]:
import os

# ── Data paths ─────────────────────────────────────────────────────────────
DATA_DIR = "/content/drive/MyDrive/llm_sft"  # sửa nếu chạy notebook ở thư mục khác (vd Colab: upload rồi trỏ lại)
CSV_FILES = [
    f"{DATA_DIR}/mapped_dataset_7label_cleaned.csv",  # cột input/output
    f"{DATA_DIR}/addon_assistant.csv",                # cột Input/Output (hoa)
    f"{DATA_DIR}/llm_assistant_fixed.csv",            # câu trả lời dài hơn ~1.6x
]
MULTITURN_FILE = f"{DATA_DIR}/hoithoaiiii.jsonl"  # TODO: đổi sang hoithoaiiii_fixed.jsonl sau khi chạy script fix

# ── Model ─────────────────────────────────────────────────────────────────
MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507"  # base model đã chốt: Qwen 4B; đổi id HF ở đây nếu cần
MAX_SEQ_LEN = 2048

# ── LoRA ──────────────────────────────────────────────────────────────────
LORA_R = 16          # rank adapter: 16 là mức cân bằng cho 5k samples
LORA_ALPHA = 32      # thường đặt = 2 * r

# ── Training ─────────────────────────────────────────────────────────────
EPOCHS = 2           # 1–2 epochs cho lần đầu; early stop theo val loss
LR = 2e-4            # học chuẩn cho LoRA
BATCH_SIZE = 4
GRAD_ACCUM = 4       # batch hiệu dụng = 4*4 = 16
VAL_RATIO = 0.03     # ~160 mẫu validation để theo dõi overfit
OUTPUT_DIR = "/content/drive/MyDrive/llm_sft/vnuf_qlora_out"
SEED = 42

## Cell 3 — Đọc & chuẩn hóa 3 CSV single-turn

3 file CSV dùng tên cột không đồng nhất (`input` vs `Input`, ...). Cell này đọc từng file, chuẩn hóa tên cột về chữ thường, lấy đúng cặp `input`/`output`, rồi đổi sang định dạng chat: mỗi mẫu là `{"messages": [user, assistant]}`. **Không đưa nhãn emotion vào input** (đã phân tích: redundant + gây leakage lúc inference).

In [5]:
import pandas as pd

def load_single_turn(path: str) -> list[dict]:
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    inp_col = next(c for c in df.columns if c in ("input", "instruction", "question"))
    out_col = next(c for c in df.columns if c in ("output", "response", "answer"))
    df = df[[inp_col, out_col]].dropna()
    return [
        {"messages": [
            {"role": "user", "content": str(i).strip()},
            {"role": "assistant", "content": str(o).strip()},
        ]}
        for i, o in zip(df[inp_col], df[out_col])
        if str(i).strip() and str(o).strip()
    ]

single_turn = []
for f in CSV_FILES:
    rows = load_single_turn(f)
    print(f"{os.path.basename(f):45s} -> {len(rows):5d} mẫu")
    single_turn.extend(rows)

# Bỏ trùng input (đã kiểm tra 0 overlap giữa các file, đây chỉ là phòng hờ)
seen, single_dedup = set(), []
for s in single_turn:
    key = s["messages"][0]["content"]
    if key not in seen:
        seen.add(key)
        single_dedup.append(s)
print(f"Tổng single-turn sau dedup: {len(single_dedup)} mẫu")

mapped_dataset_7label_cleaned.csv             ->  2653 mẫu
addon_assistant.csv                           ->  2130 mẫu
llm_assistant_fixed.csv                       ->   400 mẫu
Tổng single-turn sau dedup: 5182 mẫu


## Cell 4 — Đọc multi-turn JSONL

Mỗi dòng jsonl là 1 cuộc hội thoại `{"emotion", "style", "messages": [...]} `. Chỉ giữ `messages` (bỏ metadata emotion/style khỏi input training), kiểm tra role xen kẽ user/assistant hợp lệ.

In [6]:
import json

def load_multi_turn(path: str) -> list[dict]:
    samples = []
    with open(path, encoding="utf-8") as f:
        for ln, line in enumerate(f, 1):
            rec = json.loads(line)
            msgs = rec["messages"]
            # kiểm tra xen kẽ user/assistant, bắt đầu bằng user
            roles = [m["role"] for m in msgs]
            ok = roles[0] == "user" and all(
                roles[i] != roles[i + 1] for i in range(len(roles) - 1)
            )
            if not ok:
                print(f"  ! dòng {ln}: role không xen kẽ, bỏ qua")
                continue
            samples.append({"messages": msgs})
    return samples

multi_turn = load_multi_turn(MULTITURN_FILE)
n_turns = sum(len(s["messages"]) for s in multi_turn)
print(f"Multi-turn: {len(multi_turn)} cuộc hội thoại, ~{n_turns // 2} lượt assistant")

Multi-turn: 91 cuộc hội thoại, ~358 lượt assistant


## Cell 5 — Xem thử vài mẫu

Kiểm tra trực quan trước khi train: in 2 mẫu single-turn và 1 cuộc multi-turn. Mắt người bắt lỗi định dạng nhanh hơn bất kỳ assert nào.

In [7]:
import random
random.seed(SEED)

for s in random.sample(single_dedup, 2):
    print("USER     :", s["messages"][0]["content"][:150])
    print("ASSISTANT:", s["messages"][1]["content"][:150], "\n")

mt = random.choice(multi_turn)
print(f"--- Multi-turn mẫu ({len(mt['messages'])} messages) ---")
for m in mt["messages"]:
    print(f"{m['role']:9s}:", m["content"][:120])

USER     : Ae tư vấn giúp, điểm thi thử thấp quá, sợ thi thật sẽ tệ hơn thì nên làm sao? bất an ghê huhu
ASSISTANT: Hít thở sâu một chút nhé... Lo âu kiểu này ai cũng hiểu, đừng tự trách bản thân nhé. Tắt thông báo điện thoại một lúc, tập trung vào đúng 1 việc thôi. 

USER     : khóc không ra nước mắt nữa rồi, cảm giác trống rỗng chỉ muốn kết thúc
ASSISTANT: Mình nghe bạn. Cảm giác kiệt sức và tuyệt vọng như vậy thật nặng nề. Nhưng đây là lúc bạn cần được hỗ trợ bởi chuyên gia. Gọi 1800 599 920 — họ sẽ lắn 

--- Multi-turn mẫu (8 messages) ---
user     : mình đang tức người yêu lắm và muốn giải quyết nhưng không biết nói thế nào
assistant: Muốn giải quyết hơn là giữ trong lòng — đó là điều tốt. Bạn kể mình nghe chuyện gì đã xảy ra?
user     : họ hay hủy kế hoạch vào phút cuối, lần này là lần thứ 3 trong tháng
assistant: Ba lần trong một tháng — không phải ngẫu nhiên nữa. Mỗi lần họ hủy thì lý do là gì?
user     : hay nói bận việc hoặc mệt, nhưng mình thấy họ vẫn đi chơi với bạn bè
assi

## Cell 6 — Gộp, xáo trộn, chia train/val

Gộp 2 nguồn data, shuffle (quan trọng: tránh model thấy hết single-turn rồi mới tới multi-turn trong cùng epoch), tách ~3% làm validation. Val loss là tín hiệu chính để phát hiện overfit (val tăng → dừng).

In [8]:
from datasets import Dataset

all_samples = single_dedup + multi_turn
random.shuffle(all_samples)

ds = Dataset.from_list(all_samples)
split = ds.train_test_split(test_size=VAL_RATIO, seed=SEED)
train_ds, val_ds = split["train"], split["test"]
print(f"Train: {len(train_ds)} mẫu | Val: {len(val_ds)} mẫu")

Train: 5114 mẫu | Val: 159 mẫu


## Cell 7 — Load base model 4-bit + tokenizer

Load Qwen 4B ở chế độ 4-bit (QLoRA): trọng số gốc nén 4-bit để vừa VRAM (~4–5GB), adapter LoRA train ở fp16/bf16 phía trên. `unsloth` tự vá kernel để train nhanh + tiết kiệm VRAM hơn transformers thuần.

In [9]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
print("Đã load:", MODEL_NAME)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.6: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Đã load: unsloth/Qwen3-4B-Instruct-2507


## Cell 8 — Gắn adapter LoRA

Thay vì train toàn bộ 4 tỷ tham số (tốn VRAM, dễ overfit với 5k mẫu), chỉ train adapter LoRA nhỏ (~20–40M tham số, <1%) gắn vào các lớp attention + MLP. `use_gradient_checkpointing = "unsloth"` tiết kiệm VRAM khi backprop.

In [10]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,        # unsloth khuyên 0 để tối ưu tốc độ
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

Unsloth 2026.7.6 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


## Cell 9 — Cấu hình trainer & train

`SFTTrainer` nhận thẳng cột `messages` và tự áp chat template của Qwen (kể cả multi-turn). `dataset_text_field` không cần vì TRL mới xử lý format hội thoại tự động.

Theo dõi khi train: **train loss giảm đều, eval loss (val) chạm đáy rồi tăng = dấu hiệu overfit → dừng**. Với cấu hình này 2 epochs thường ổn; nếu val loss tăng sớm, chạy lại với `EPOCHS = 1`.

*(Chi tiết kỹ thuật: mặc định loss tính trên cả user + assistant tokens — đủ dùng cho lần đầu; muốn chỉ tính loss trên câu trả lời assistant thì thêm `assistant_only_loss=True` nếu chat template hỗ trợ.)*

In [20]:
def format_messages(example):
      msgs = example["messages"]
      if isinstance(msgs, dict):
          msgs = [msgs]  # bọc dict thành list
      example["messages"] = msgs
      return example

train_ds = train_ds.map(format_messages)
val_ds = val_ds.map(format_messages)

Map:   0%|          | 0/5114 [00:00<?, ? examples/s]

Map:   0%|          | 0/159 [00:00<?, ? examples/s]

In [24]:
from trl import SFTConfig, SFTTrainer

# Unsloth mới yêu cầu formatting_func
def formatting_func(examples):
      convs = examples["messages"]
      # unbatched: convs là 1 conversation (list of dicts) → bọc thành [convs]
      if convs and isinstance(convs[0], dict):
          convs = [convs]
      return [
          tokenizer.apply_chat_template(
              conv, tokenize=False, add_generation_prompt=False
          )
          for conv in convs
      ]

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    formatting_func=formatting_func,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        max_length=MAX_SEQ_LEN,
        warmup_steps=5,          # thay thế warmup_ratio bị deprecated
        lr_scheduler_type="cosine",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="epoch",
        seed=SEED,
        report_to="none",
    ),
)

trainer.train()


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/5114 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/159 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,114 | Num Epochs = 2 | Total steps = 640
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
50,1.195562,1.106407
100,0.779065,0.710519
150,0.668261,0.601316
200,0.616334,0.553748
250,0.515313,0.520267
300,0.548140,0.498154
350,0.488296,0.482616
400,0.429268,0.473818
450,0.512336,0.462147
500,0.457838,0.455796


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/llm_sft/vnuf_qlora_out/checkpoint-320/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/llm_sft/vnuf_qlora_out/checkpoint-640/tokenizer_config.json.


TrainOutput(global_step=640, training_loss=0.678531889617443, metrics={'train_runtime': 2857.3592, 'train_samples_per_second': 3.58, 'train_steps_per_second': 0.224, 'total_flos': 1.88018811245568e+16, 'train_loss': 0.678531889617443, 'epoch': 2.0})

## Cell 10 — Thử generate vài câu

Sau train, kiểm tra tay bằng vài câu hỏi thật, gồm cả trường hợp **multi-turn** (cho sẵn lịch sử hội thoại) để xem model có giữ mạch không. Đây là bước quan trọng nhất để phát hiện học vẹt template — nếu mọi câu trả lời đều mở đầu giống nhau, data cần đa dạng hơn.

In [25]:
FastLanguageModel.for_inference(model)  # bật chế độ inference nhanh

def chat(messages: list[dict], max_new_tokens: int = 256) -> str:
    ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    out = model.generate(ids, max_new_tokens=max_new_tokens, temperature=0.7, top_p=0.9)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

# 1. Single-turn
print(chat([{"role": "user", "content": "mình đang rất mệt mỏi, công việc áp lực quá"}]))
print("=" * 60)

# 2. Multi-turn: model phải nhớ người dùng vừa nói về sếp mới
print(chat([
    {"role": "user", "content": "dạo này mình mất ngủ nhiều"},
    {"role": "assistant", "content": "Mất ngủ kéo dài thật sự kiệt sức. Bạn mất ngủ từ khi nào, có chuyện gì xảy ra trước đó không?"},
    {"role": "user", "content": "từ khi sếp mới về, công việc căng thẳng hẳn"},
]))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn

Mệt mỏi do áp lực công việc là điều dễ hiểu, bạn đang làm gì?
Sếp mới và áp lực công việc — hai thứ đôi khi không hòa thuận. Bạn đang làm công việc gì và sếp mới thường kiểm soát điều gì?


## Cell 11 — Lưu adapter

Lưu adapter LoRA (vài chục MB) — đây là artifact chính mang đi deploy. Tuỳ chọn merge vào base model thành model đầy đủ (`merged_16bit`) nếu muốn serve trực tiếp bằng vLLM/transformers mà không cần peft.

In [26]:
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
print("Đã lưu adapter tại", f"{OUTPUT_DIR}/lora_adapter")

# Tuỳ chọn: merge thành model đầy đủ (~8GB) — bỏ comment nếu cần
# model.save_pretrained_merged(f"{OUTPUT_DIR}/merged_16bit", tokenizer, save_method="merged_16bit")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/llm_sft/vnuf_qlora_out/lora_adapter/tokenizer_config.json.


Đã lưu adapter tại /content/drive/MyDrive/llm_sft/vnuf_qlora_out/lora_adapter


## Bước tiếp theo

1. Nếu kết quả generate ổn → gắn adapter vào `backend/assistant_service.py`, thay template responses bằng model.
2. Nếu trả lời lặp template / học vẹt → giảm epochs xuống 1, hoặc xem lại độ đa dạng của data.
3. Nếu multi-turn yếu → thu thập thêm hội thoại thật ở vòng data tiếp theo.